In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

A subset of the molecules should be computed by Boltz-2. 

Selections are stratified across potency ranges. RDKit `MaxMinPicker` is applied on the fingerprints so the subset is scaffold-diverse. 

In [2]:
import pandas as pd
from rdkit.Chem import PandasTools

df = pd.read_csv(DATA_PROCESSED / "egfr_clean_classified.csv")
PandasTools.AddMoleculeColumnToFrame(df, smilesCol="Clean Smiles", molCol="RDkit Mol", includeFingerprints=True)

In [3]:
# Stratify by potentcy
import numpy as np

df["Potency"] = pd.cut(df["pChEMBL Median"], 
                        bins=[-np.inf,6,8,np.inf],
                        labels=["low", "mid", "high"])
df["Potency"].value_counts()

Potency
mid     3500
low     1934
high    1275
Name: count, dtype: int64

In [4]:
# Check cross tabulation of potency x class
ct = pd.crosstab(df["Potency"], df["Classification"])
ct

Classification,4-Anilinoquinazoline-like,Aminopurine-like,Covalent,Monocyclic,Other
Potency,,,,,
low,206,3,410,46,1269
mid,938,64,991,18,1489
high,433,94,437,0,311


In [5]:
print(ct.to_markdown())

| Potency   |   4-Anilinoquinazoline-like |   Aminopurine-like |   Covalent |   Monocyclic |   Other |
|:----------|----------------------------:|-------------------:|-----------:|-------------:|--------:|
| low       |                         206 |                  3 |        410 |           46 |    1269 |
| mid       |                         938 |                 64 |        991 |           18 |    1489 |
| high      |                         433 |                 94 |        437 |            0 |     311 |


There is a moderate indication of potency, relative to manual classification. 
It is therefore wise to check that the stratification picks compounds within 
each potency bin from a spread of chemotypes. The previous notebook already 
establishes correlation between fingerprint and classification, so applying 
`MinMaxPicker` over the fingerprints should account for this.

In [6]:
# Use MinMaxPicker over the potency bins; 15 molecules per bin
from rdkit.Chem import rdFingerprintGenerator
from rdkit.SimDivFilters import rdSimDivPickers

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
picker = rdSimDivPickers.MaxMinPicker()

selected = []
for label in df["Potency"].unique():
    sub = df[df["Potency"] == label].reset_index(drop=True)
    # Compute fingerprints
    sub_fps = [gen.GetFingerprint(m) for m in sub["RDkit Mol"]]
    picks = picker.LazyBitVectorPick(sub_fps, len(sub_fps), 15, seed=42)
    selected.append(sub.iloc[list(picks)])

subset = pd.concat(selected, ignore_index=True)

### Binding Type classification

1. type I inhibitors bind to an active enzyme conformation.
(1.5. type I½ inhibitors
as drugs that bind to a dormant protein kinase with the DFG-D directed inward (DFG-D in) toward the active site)
2. type II inhibitors bind to a dormant protein kinase with the
DFG-D of the activation segment pointing away from the active site
(DFG-$D_{\text{out}}$)
3. type III allosteric inhibitors bind adjacently to the adenine binding pocket
4. type IV allosteric inhibitors bind elsewhere
5. type V bivalent inhibitors span two distinct parts of the protein kinase domain 
6. type VI inhibitors generally form a covalent adduct (covalent bond with EGFR C979)

ref: `refs/Roskoski-2019-Small-molecule-inhibitors-targeting-the-EGFR-ErbB-family-of-protein-tyrosine-kinases-in-human-cancers.pdf`

In [7]:
# Construct table by manually referencing the data, see Roskoski 2019
drugs = [
    {"Drug Name" : "Gefitinib", "PubMed CID" : "123631", "Binding Type" : "type I", "Complex PDB" : "2ITY", "Description" : "quinazoline derivative"},
    {"Drug Name" : "Erlotinib", "PubMed CID" : "176870", "Binding Type" : "type II", "Complex PDB" : "1M17", "Description" : "quinazoline derivative"},
    {"Drug Name" : "Osimertinib", "PubMed CID" : "71496458", "Binding Type" : "type VI", "Complex PDB" : "4ZAU", "Description" : "indole-pyrimidine derivative"},
    {"Drug Name" : "Lapatinib", "PubMed CID" : "208908", "Binding Type" : "type I.5", "Complex PDB" : "1XKK", "Description" : "chlorophenyl-quinazoline"},
    {"Drug Name" : "Avitinib", "PubMed CID" : "72734520", "Binding Type" : "type VI", "Complex PDB" : None, "Description" : "pyrrolopyrimidine"},
    {"Drug Name" : "Olmutinib", "PubMed CID" : "54758501", "Binding Type" : "type VI", "Complex PDB" : None, "Description" : "anilino-thienopyrimidine derivative"},
    {"Drug Name" : "Pelitinib", "PubMed CID" : "6445562", "Binding Type" : "type VI", "Complex PDB" : None, "Description" : "fluroanilino-quinoline derivative"},
]
df_drugs = pd.DataFrame(drugs)

In [8]:
# Get SMILES of the drugs, using CID
import requests
from rdkit import Chem

def cid_to_smi(cid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/TXT"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    smi = r.text.strip()
    return smi

df_drugs["Smiles"] = df_drugs["PubMed CID"].apply(cid_to_smi)

In [9]:
# Clean the SMILES
# Standardization function from the chembl_data notebook
import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from standardize import standardize_mols
%reload_ext autoreload

clean_reps = df_drugs["Smiles"].apply(standardize_mols)
df_drugs["Clean Smiles"] = clean_reps.str[0]
df_drugs["Clean InchiKey"] = clean_reps.str[1]

### Known drugs, targeting EGFR

7 well-known drugs targeting EGFR are added to the subset. They span 4 binding types (type I, type II, type $1\tfrac{1}{2}$ and type VI (covalent); allosteric binding types are not included). A reference complex is available from the PDB for each binding type. Three of the drugs are covalent binders with no available experimental structure.

In [10]:
df_drugs

,Drug Name,PubMed CID,Binding Type,Complex PDB,Description,Smiles,Clean Smiles,Clean InchiKey
0,Gefitinib,123631,type I,2ITY,quinazoline derivative,COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)F)Cl)OC...,COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1,XGALLCVXEZPNRQ-UHFFFAOYSA-N
1,Erlotinib,176870,type II,1M17,quinazoline derivative,COCCOC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=CC(=C3)C#C...,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG-UHFFFAOYSA-N
2,Osimertinib,71496458,type VI,4ZAU,indole-pyrimidine derivative,CN1C=C(C2=CC=CC=C21)C3=NC(=NC=C3)NC4=C(C=C(C(=...,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(O...,DUYJMQONPNNFPI-UHFFFAOYSA-N
3,Lapatinib,208908,type I.5,1XKK,chlorophenyl-quinazoline,CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3...,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,BCFGMOOMADDAQU-UHFFFAOYSA-N
4,Avitinib,72734520,type VI,NaN,pyrrolopyrimidine,CN1CCN(CC1)C2=C(C=C(C=C2)NC3=NC4=C(C=CN4)C(=N3...,C=CC(=O)Nc1cccc(Oc2nc(Nc3ccc(N4CCN(C)CC4)c(F)c...,UOFYSRZSLXWIQB-UHFFFAOYSA-N
5,Olmutinib,54758501,type VI,NaN,anilino-thienopyrimidine derivative,CN1CCN(CC1)C2=CC=C(C=C2)NC3=NC4=C(C(=N3)OC5=CC...,C=CC(=O)Nc1cccc(Oc2nc(Nc3ccc(N4CCN(C)CC4)cc3)n...,FDMQDKQUTRLUBU-UHFFFAOYSA-N
6,Pelitinib,6445562,type VI,NaN,fluroanilino-quinoline derivative,CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)F)Cl)...,CCOc1cc2ncc(C#N)c(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)...,WVUNYSQLFKLYNI-UHFFFAOYSA-N


In [11]:
len(subset)

45

In [12]:
# Check if the drug is already in the subset
for d in df_drugs["Clean InchiKey"]:
    print(d in df["Clean InchiKey"].values)

# Get drug's indicies from the full df
# Add a label for ref compound columns
matches = df[df["Clean InchiKey"].isin(df_drugs["Clean InchiKey"])]
df["Known Drug"] = False
subset["Known Drug"] = False
matches["Known Drug"] = True
subset = pd.concat([subset, matches], ignore_index=True)

True
True
True
True
True
True
False


In [13]:
# One wasn't added in
missing = df_drugs[~df_drugs["Clean InchiKey"].isin(df["Clean InchiKey"])]
missing

,Drug Name,PubMed CID,Binding Type,Complex PDB,Description,Smiles,Clean Smiles,Clean InchiKey
6,Pelitinib,6445562,type VI,NaN,fluroanilino-quinoline derivative,CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)F)Cl)...,CCOc1cc2ncc(C#N)c(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)...,WVUNYSQLFKLYNI-UHFFFAOYSA-N


Pelitinib is missing, probably due to a mismatch in the InchiKeys. It's a good idea to check the standardization workflow, because something may be going wrong with respect to canonicalization. For now, Pelitinib is left excluded from the subset, since there are already 3 other covalent binders and there is no experimental structure of the complex.

### Decoy inclusion

inactives are also used so that the set as inactives. Often DUD-E is used to generate some presumed nonbinders, but might be separated by a model on physicochemical properties alone. In this case however, it may be advantageous to use binders with *very* low activities. This approach uses known, rather than presumed, data. While these inactives aren't true non-binders, separating weak from potent binders is a stronger benchmarking property.

Property matched inactives (by MW, cLogP, etc.) are used so that the model doesn't predict nonbinding purely from the chemotype properties. The difference between inactives and binders should some down to structure. 

In [14]:
very_weak = df[df["pChEMBL Median"] < 4.5]
actives = df[df["pChEMBL Median"] > 8]
len(very_weak)

230

In [15]:
# Properties to match on
props = ["Molecular Weight", "cLogP", "Polar Surface Area",
         "Hydrogen Bond Donors", "Hydrogen Bond Acceptors", "Num Rotatable Bonds"]

In [16]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

# Standardize the scale of properties
scaler = StandardScaler().fit(df[props])
A = scaler.transform(actives[props])
P = scaler.transform(very_weak[props])

# Compute property distances for each pair
dist = cdist(A, P) 

In [17]:
# Pick 50 inactives
import numpy as np
import random

# Reproducible
random.seed(92)

chosen = []
used = set()
# for 20 random actives...
for i in random.sample(range(len(actives)), 50):
    # find the nearest, unused nonbinder
    order = np.argsort(dist[i])
    for j in order:
        if j not in used:
            used.add(j)
            chosen.append(j)
            break

inactives = very_weak.iloc[chosen]

In [18]:
# Remove inactives that are structurally too similar (no analogues)
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Compute fingerprints
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
actives_fps = [gen.GetFingerprint(a) for a in actives["RDkit Mol"]] 
inactives_fps = [gen.GetFingerprint(d) for d in inactives["RDkit Mol"]]

# Compute Tanimoto similarity
sim = np.array([DataStructs.BulkTanimotoSimilarity(inactive_fp, actives_fps) for inactive_fp in inactives_fps])

In [19]:
# Drop inactives with similarities > 0.4 to any active
inds = np.max(sim, axis=1) < 0.4
inactives = inactives[inds]

# Then randomly select 10
inactives = inactives.sample(n=10, random_state=92)
len(inactives)

10

In [20]:
# Add a label for decoy columns
subset["Decoy"] = False
inactives["Decoy"] = True

# Add inactives to the subset
subset = pd.concat([subset, inactives], ignore_index=True)
subset = subset.drop_duplicates("Clean InchiKey")
len(subset)

61

In [21]:
print(subset["Decoy"].value_counts())
print(len(subset), "compounds total")
print(pd.crosstab(subset["Potency"], subset["Decoy"]))

Decoy
False    51
True     10
Name: count, dtype: int64
61 compounds total
Decoy    False  True 
Potency              
low         15     10
mid         21      0
high        15      0


In [22]:
# Write to file
cols = [col for col in subset.columns if "RDkit Mol" != col]
subset.to_csv(DATA_PROCESSED / "boltz_subset.csv", columns=cols, index=False)

In [23]:
# Write reference table to file
df_drugs.to_csv(DATA_PROCESSED / "known_refs.csv", index=False)

In [24]:
subset

,Clean InchiKey,Clean Smiles,Molecule ChEMBL ID,Acrylamide,pChEMBL Median,pChEMBL std,Activity Measurements,Molecular Weight,cLogP,Polar Surface Area,...,Aromatic Rings,QED,Lipinski Violations,Ligand Efficiency,Murcko Scaffold,Classification,RDkit Mol,Potency,Known Drug,Decoy
0,BULITYBNVIHSDM-UHFFFAOYSA-N,C=CC(=O)Nc1cccc(Oc2nc(Nc3ccc(N4CCN(S(C)(=O)=O)...,CHEMBL3797963,True,6.14,NaN,1,570.697,3.94910,125.99,...,3,0.273336,1,0.215687,c1ccc(Oc2ccnc(Nc3ccc(N4CCNCC4)cc3)n2)cc1,Covalent,<rdkit.Chem.rdchem.Mol object at 0x000001CDD1C...,mid,False,False
1,ACGUYXCXAPNIKK-UHFFFAOYSA-N,Oc1c(Cl)cc(Cl)c(Cl)c1Cc1c(O)c(Cl)cc(Cl)c1Cl,CHEMBL496,False,6.13,NaN,1,406.907,6.60900,40.46,...,2,0.550693,1,0.399910,c1ccc(Cc2ccccc2)cc1,Other,<rdkit.Chem.rdchem.Mol object at 0x000001CDD4C...,mid,False,False
2,NFJFOCDCCGFTGX-CXUHLZMHSA-N,O=[N+]([O-])c1ccc(/C=N/Nc2nnc(-c3ccncc3)o2)cc1,CHEMBL4561572,False,6.38,NaN,1,310.273,2.48580,119.34,...,3,0.436401,0,0.380026,C(=N\Nc1nnc(-c2ccncc2)o1)/c1ccccc1,Other,<rdkit.Chem.rdchem.Mol object at 0x000001CDD1B...,mid,False,False
3,NCQYFKGTEDNHPE-UHFFFAOYSA-N,c1ccc(CNc2ncnc3c2[nH]c2ccccc23)cc1,CHEMBL5173075,False,6.34,NaN,1,274.327,3.72320,53.60,...,4,0.598326,0,0.413610,c1ccc(CNc2ncnc3c2[nH]c2ccccc23)cc1,Other,<rdkit.Chem.rdchem.Mol object at 0x000001CDD4C...,mid,False,False
4,VRJQXKKOPVMHKL-DHMZDEOHSA-N,C/C=C(\C)C(O[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)...,CHEMBL1817964,False,6.80,0.251595,3,562.700,3.43404,138.82,...,1,0.222873,1,0.232900,O=c1ccoc(CC=CC/C=C/C=C/CCO[C@H]2CCCCO2)c1,Monocyclic,<rdkit.Chem.rdchem.Mol object at 0x000001CDD0B...,mid,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,WYNQMPRRXLNUQG-MOSHPQCFSA-N,COc1cc(/C=C2\SC(=N)NC2=O)ccc1OCCCOc1cc(C)cc(C)c1,CHEMBL602890,True,4.25,NaN,1,412.511,4.29861,80.64,...,2,0.499135,0,0.200776,N=C1NC(=O)/C(=C/c2ccc(OCCCOc3ccccc3)cc2)S1,Covalent,<rdkit.Chem.rdchem.Mol object at 0x000001CDD4C...,low,False,True
57,WFFWDHORGIOMIR-UHFFFAOYSA-N,O=[N+]([O-])CCc1c(SSc2[nH]c3ccccc3c2CC[N+](=O)...,CHEMBL158119,False,4.05,NaN,1,442.522,5.08700,117.86,...,4,0.211797,1,0.184950,c1ccc2[nH]c(SSc3cc4ccccc4[nH]3)cc2c1,Other,<rdkit.Chem.rdchem.Mol object at 0x000001CDD4C...,low,False,True
58,JKFICPIMSFBOJJ-UHFFFAOYSA-N,Cc1ccc(N2NC(=O)C(=Cc3ccc(-c4cc([N+](=O)[O-])cc...,CHEMBL2029690,True,4.13,NaN,1,419.393,3.63864,125.92,...,3,0.287775,0,0.182519,O=C1NN(c2ccccc2)C(=O)C1=Cc1ccc(-c2ccccc2)o1,Covalent,<rdkit.Chem.rdchem.Mol object at 0x000001CDD4C...,low,False,True
59,BWFSEWLGHWAHGL-UHFFFAOYSA-N,N#CCCc1c(SSc2[nH]c3ccccc3c2CCC#N)[nH]c2ccccc12,CHEMBL157304,False,4.33,NaN,1,402.548,6.36096,79.16,...,4,0.352457,1,0.211861,c1ccc2[nH]c(SSc3cc4ccccc4[nH]3)cc2c1,Other,<rdkit.Chem.rdchem.Mol object at 0x000001CDD0B...,low,False,True
